In [1]:
# Instalación de dependencias.
# Colab trae TensorFlow 2.16+ con Keras 3 por defecto, pero transformers
# necesita TFBertModel (Keras 2). Instalamos transformers fijado a una
# versión compatible + tf-keras (shim de Keras 2) y forzamos su uso en
# TODO el notebook (no solo en transformers) para evitar mezclar tensores
# de Keras 3 (KerasTensor) con capas que esperan Keras 2.
!pip install -q "transformers==4.40.0" "tf-keras" openpyxl


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [2]:
!pip uninstall -y hf_xet

Found existing installation: hf-xet 1.5.1
Uninstalling hf-xet-1.5.1:
  Successfully uninstalled hf-xet-1.5.1


In [3]:
# Montar Google Drive (opcional pero recomendado para no perder archivos al cerrar la sesión)
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import numpy as np
import pandas as pd
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf
import tf_keras as keras
from tf_keras.layers import Input, Bidirectional, LSTM, Dropout, GlobalAveragePooling1D, Dense
from tf_keras.models import Model
from tf_keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [5]:
np.random.seed(1234)
tf.random.set_seed(1234)

In [6]:
print("tf_keras version:", keras.__version__)

tf_keras version: 2.20.0


In [7]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [8]:
!rm -rf ~/.cache/huggingface

In [9]:
"""# Import and split training data"""
df = pd.read_csv("/content/drive/MyDrive/TFM/FED_classification.csv", sep='\t')
df['cat']=df['sentiment'].apply(lambda x: 2 if x=='dovish' else (0 if x=='hawkish' else 1))

# create balanced sample
hawkish = df[df['sentiment']=='hawkish']
dovish = df[df['sentiment']=='dovish']
neutral = df[df['sentiment']=='neutral']
dovish_downsampled = dovish.sample(hawkish.shape[0])
neutral_downsampled = neutral.sample(hawkish.shape[0])
df_balanced = pd.concat([dovish_downsampled,neutral_downsampled,hawkish])

train, remain = train_test_split(df_balanced, stratify=df_balanced['cat'], train_size=0.8)
test, val = train_test_split(remain, stratify=remain['cat'], train_size=0.5)


In [10]:
# Pre-processing data Tokenize and store results in arrays to add in dataset later***
# initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = TFBertModel.from_pretrained('bert-base-uncased', from_pt=True)
def tokenize(data):
    seq_len = 256
    num_samples = len(data)

    # initialize empty zero arrays
    Xids = np.zeros((num_samples, seq_len))
    Xmask = np.zeros((num_samples, seq_len))

    for i, sentence in enumerate(data['text']):
        tokens = tokenizer.encode_plus(sentence, max_length=seq_len, truncation=True,
                                       padding='max_length', add_special_tokens=True,
                                       return_tensors='tf')
        # assign tokenized outputs to respective rows in numpy arrays
        Xids[i, :] = tokens['input_ids']
        Xmask[i, :] = tokens['attention_mask']
    # check shape
    print(Xids.shape)
    return Xids, Xmask

Xids_train, Xmask_train = tokenize(train)
Xids_val, Xmask_val = tokenize(val)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already

(583, 256)
(73, 256)


## Precalcular embeddings de BERT (clave para la velocidad)

BERT esta congelado (`trainable=False` mas abajo): sus pesos nunca cambian
durante el entrenamiento. Aqui calculamos su salida UNA sola vez para train
y val, y entrenamos solo el BiLSTM+Dense sobre esos embeddings ya
calculados, en vez de recalcular BERT en cada batch de cada una de las 200
epochs (que es lo que tardo 5 horas para solo 18 epochs). Matematicamente
identico a lo que tenias, solo que sin repetir un calculo que no cambia.

In [ ]:
def compute_bert_embeddings(Xids, Xmask, batch_size=8, verbose_every=20):
    """Pasa el corpus por BERT UNA vez y devuelve el last_hidden_state.
    BERT esta congelado, asi que este resultado no cambia entre epochs."""
    n = Xids.shape[0]
    seq_len = Xids.shape[1]
    embs = np.zeros((n, seq_len, 768), dtype='float32')
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_ids = tf.cast(Xids[start:end], tf.int32)
        batch_mask = tf.cast(Xmask[start:end], tf.int32)
        out = bert.bert(batch_ids, attention_mask=batch_mask)[0]
        embs[start:end] = out.numpy()
        if (start // batch_size) % verbose_every == 0:
            print(f"  embeddings {end}/{n}")
    return embs

print("Precalculando embeddings BERT (train)...")
Emb_train = compute_bert_embeddings(Xids_train, Xmask_train, batch_size=8)
print("Precalculando embeddings BERT (val)...")
Emb_val = compute_bert_embeddings(Xids_val, Xmask_val, batch_size=8)

import gc
del Xids_train, Xmask_train, Xids_val, Xmask_val
gc.collect()

In [11]:
# This tokenize function is to process testing data later
def prep_data(text):
    tokens = tokenizer.encode_plus(text, max_length=256, truncation=True,
                                   padding='max_length', add_special_tokens=True,
                                   return_token_type_ids=False,
                                   return_tensors='tf')
    # tokenizer returns int32 tensors, we need to return float64, so we use tf.cast
    return {'input_ids': tf.cast(tokens['input_ids'], tf.float64),
            'attention_mask': tf.cast(tokens['attention_mask'], tf.float64)}

In [12]:
# One-hot encoding the sentiment
labels_train = LabelBinarizer().fit_transform(train.cat)
labels_val = LabelBinarizer().fit_transform(val.cat)


"""
Build 2 tf.data.Dataset objects (train & test) using tokenized outputs and label tensors.
Then transform into the correct format for keras input layers
"""


In [13]:
dataset_train = tf.data.Dataset.from_tensor_slices((Emb_train, labels_train))
dataset_val = tf.data.Dataset.from_tensor_slices((Emb_val, labels_val))

"""**Batch and shuffle data**"""

In [14]:
# El modelo ya no incluye a BERT dentro del grafo de entrenamiento (se
# calculo aparte, una vez), asi que podemos permitirnos un batch mayor.
batch_size = 8

In [15]:
# shuffle and batch
train_ds = dataset_train.shuffle(100).batch(batch_size, drop_remainder=False)
val_ds = dataset_val.shuffle(100).batch(batch_size, drop_remainder=False)


"""# Build and Train"""

"""
    Build a network of which input is the output from BERT

"""

In [16]:
emb_input = Input(shape=(Emb_train.shape[1], 768), name='bert_embeddings')

In [ ]:
# use BERT outut as input for the next layers
net = Bidirectional(LSTM(256, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))(emb_input)
net = Dropout(0.1)(net)
net = GlobalAveragePooling1D()(net)
net = Dropout(0.1)(net)
net = Dense(256, activation='relu')(net)
net = Dropout(0.1)(net)
net = Dense(128, activation='relu')(net)
out = Dense(3, activation='softmax', name='outputs')(net)

"""
    Initialize model and train
"""

In [ ]:
model = Model(inputs=emb_input, outputs=out)
# Ya no hace falta "model.layers[2].trainable = False": BERT no esta dentro
# de este grafo (se ejecuto fuera, una vez, como extractor fijo). El efecto
# de "congelado" es el mismo.

model_path="/content/drive/MyDrive/TFM/"
optimizer = keras.optimizers.Adam()
loss = keras.losses.CategoricalCrossentropy()
acc = keras.metrics.CategoricalAccuracy('accuracy')
precision = keras.metrics.Precision()
recall = keras.metrics.Recall()

model.compile(optimizer=optimizer,
              loss=loss,
              metrics=[acc, precision, recall])

# EarlyStopping: para en cuanto val_loss deje de mejorar, en vez de forzar
# las 200 epochs completas. ModelCheckpoint: guarda el mejor modelo por si
# la sesion se cae a mitad de entrenamiento.
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint(model_path + 'text_model_ckpt', monitor='val_loss',
                             save_best_only=True, save_format='tf')

model_training = model.fit(train_ds,
                           validation_data = val_ds,
                           epochs=50,   # techo; EarlyStopping normalmente parara antes
                           callbacks=[early_stop, checkpoint])

model.save(model_path+'text_model', save_format='tf')

"""
    Testing
"""

In [ ]:
Xids_test, Xmask_test = tokenize(test)
Emb_test = compute_bert_embeddings(Xids_test, Xmask_test, batch_size=8)
probs_test = model.predict(Emb_test, batch_size=8)
test['pred'] = probs_test.argmax(axis=1)

print(classification_report(test.cat.tolist(), test.pred.tolist()))

# Aplicar el modelo al corpus del BCE  → Text_m

El bloque de arriba entrena con la Fed y testea con la Fed. Aqui aplicamos el
modelo ya entrenado (mismo `model`, misma sesion) al corpus del BCE, parrafo a
parrafo, igual que hace el pipeline de FinBERT. Misma tokenizacion que en
`tokenize` (max_length=256, truncation=True): la unica truncacion es la nativa
de 256 tokens (~180-200 palabras) -- una desviacion real respecto a los ~300
palabras de los chunks originales, a mencionar como limitacion.

Convencion de clases (de `cat`): 0 = hawkish, 1 = neutral, 2 = dovish.

In [ ]:
# Rutas (ajustadas a tu Drive)
ECB_CORPUS = "/content/drive/MyDrive/TFM/ECB_TEXT__2013-2024__v2.xlsx"
OUT_XLSX   = "/content/drive/MyDrive/TFM/output_textmodel_ECB_v2.xlsx"            # nivel parrafo
OUT_CONF   = "/content/drive/MyDrive/TFM/output_textmodel_ECB_v2_conference.xlsx" # nivel conferencia (Text_m)

LABELS = {0: "hawkish", 1: "neutral", 2: "dovish"}
ECB_SEQ_LEN = 256  # igual que en el entrenamiento

In [ ]:
def predict_paragraphs(texts, batch_size=8):
    """Tokeniza, calcula embeddings BERT una vez, y predice con el modelo entrenado."""
    n = len(texts)
    Xids = np.zeros((n, ECB_SEQ_LEN))
    Xmask = np.zeros((n, ECB_SEQ_LEN))
    for i, sentence in enumerate(texts):
        tok = tokenizer.encode_plus(str(sentence), max_length=ECB_SEQ_LEN, truncation=True,
                                    padding='max_length', add_special_tokens=True,
                                    return_tensors='tf')
        Xids[i, :] = tok['input_ids']
        Xmask[i, :] = tok['attention_mask']
    embs = compute_bert_embeddings(Xids, Xmask, batch_size=batch_size)
    probs = model.predict(embs, batch_size=batch_size, verbose=1)
    return probs


def classify_sheet(df_para):
    """Devuelve la hoja de parrafos con pred + probabilidades por clase."""
    df_out = df_para.copy()
    probs = predict_paragraphs(df_out['text'].tolist())
    df_out['pred'] = probs.argmax(axis=1)
    df_out['label'] = df_out['pred'].map(LABELS)
    df_out['p_hawkish'] = probs[:, 0]
    df_out['p_neutral'] = probs[:, 1]
    df_out['p_dovish'] = probs[:, 2]
    df_out['n_words'] = df_out['text'].astype(str).str.split().str.len()
    return df_out.drop(columns=['text'])

In [ ]:
# Cargar hojas de párrafos del corpus del BCE
remarks_para = pd.read_excel(ECB_CORPUS, sheet_name='remarks_para')  # date, order, text
answers_para = pd.read_excel(ECB_CORPUS, sheet_name='answers_para')  # date, item, order, speaker, text

# mantener `date` como TEXTO (evita la corrupción por autofill de Excel)
remarks_para['date'] = remarks_para['date'].astype(str)
answers_para['date'] = answers_para['date'].astype(str)

print("Clasificando remarks (texto preparado)...")
remarks_pred = classify_sheet(remarks_para)
print("Clasificando answers (Q&A)...")
answers_pred = classify_sheet(answers_para)

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as w:
    remarks_pred.to_excel(w, sheet_name='remarks_textmodel', index=False)
    answers_pred.to_excel(w, sheet_name='answers_textmodel', index=False)
print("Guardado (nivel párrafo) ->", OUT_XLSX)

## Tono a nivel conferencia  (Text_m)

Índice de tono de Gorodnichenko:  `tone = (#hawkish − #dovish) / N`  sobre las
predicciones de párrafo. Tres series:
`w_tone` (remarks preparados), `qa_tone` (respuestas Q&A), `all_tone` (combinado).

In [ ]:
def tone_by_date(df_pred):
    g = df_pred.groupby('date')['pred']
    n_hawk = g.apply(lambda s: (s == 0).sum())
    n_neu  = g.apply(lambda s: (s == 1).sum())
    n_dov  = g.apply(lambda s: (s == 2).sum())
    n_tot  = g.size()
    out = pd.DataFrame({'n_hawkish': n_hawk, 'n_neutral': n_neu, 'n_dovish': n_dov,
                        'n_paras': n_tot, 'tone': (n_hawk - n_dov) / n_tot})
    return out.reset_index()

w_tone  = tone_by_date(remarks_pred).rename(columns={'tone': 'w_tone'})
qa_tone = tone_by_date(answers_pred).rename(columns={'tone': 'qa_tone'})
all_pred = pd.concat([remarks_pred[['date', 'pred']], answers_pred[['date', 'pred']]], ignore_index=True)
all_tone = tone_by_date(all_pred).rename(columns={'tone': 'all_tone'})

conf = (w_tone[['date', 'w_tone']]
        .merge(qa_tone[['date', 'qa_tone']], on='date', how='outer')
        .merge(all_tone[['date', 'all_tone']], on='date', how='outer')
        .sort_values('date'))

conf.to_excel(OUT_CONF, index=False, engine='openpyxl')
print("Guardado (nivel conferencia, Text_m) ->", OUT_CONF)
print(conf.shape)   # (104, 4) esperado
conf.head()

# OJO: no abras OUT_XLSX / OUT_CONF en Excel — inspecciona con pandas (regla de la fecha).